# Future research: from system boundaries to research routes

> Earlier lectures took the Agent from a single LLM call to a system that can verify, use tools, plan, train, self-evolve, and enter the physical world. Looking back, every system shares one skeleton: generate candidates, check quality, and feed **feedback** back into the loop.
>
> This lecture is a close plus a look ahead. We first measure how long a task current systems can complete reliably — with a ruler called **time horizon**; then we sort the remaining stuck points into five classes as an open-problem inventory; finally we write, for each class, a research question that can be checked by hand, each starting from a minimal version already implemented in this course.

Let an Agent run continuously for 6 hours on a data analysis and write a report.

First, it starts from data cleaning, and every step still looks normal.

Second, somewhere in the middle a step goes off course — perhaps a column was skipped, or a field was misunderstood.

Third, every later step is built on that deviation: wrong inputs produce wrong intermediate results, which are fed to the next step, the run drifts further, and the report is wrong from beginning to end.

The numbers METR measured with the "time horizon" ruler tell the same story: Opus 4.5 can complete about 4 hours 49 minutes of work at 50% success, but when the success requirement is raised to 80%, the tasks it can complete reliably shrink to about 27 minutes. The drop between 50% and 80% is the **reliability gap**, the structural reason long tasks are hard to stabilize. Long-horizon **autonomy** is still an open problem.

Put the systems built in the previous 16 lectures side by side and they share one skeleton: generate candidates, check quality, feed feedback back into the loop. The variously named concepts are instances of that skeleton: test-time compute lets the loop generate several times and then choose, a verifier makes the check accurate, ReAct lets environment feedback enter the loop, tree search adds forks to the loop, RL scaling writes every ring of the loop into weights, open-ended evolution lets the model modify the loop itself. Capabilities differ; the skeleton is the same.

The skeleton runs far on the course's benchmarks, but scaling it to longer tasks, more Agents, and the real physical world hits questions that currently have no answer. Questions with no answer that are still worth doing are called **open problems**. This lecture starts from the boundary, and uses one shared ruler to measure how much stronger models have become in recent years.


## 1. Boundaries of current systems

This section locates the systems we can now build after 16 lectures. We line up the 16 functional nodes in course order and see how they stack.

Single-shot generation is the starting point: the model receives a task and emits an answer. Every later lecture wraps another loop around the previous layer. First comes verification, so the model's output can be checked; then tool use, so the check does not rely on guessing and can actually run in the environment; then planning, so a long task is split into steps; then training, so the order of work is written into model weights; then self-evolution, so the model can modify the logic of its own judgments. By Lecture 16 this loop is already connected to the physical world and can operate a real robot.

These 16 nodes are not 16 unrelated features; they are 16 instances of the same skeleton, and the skeleton is generate, verify, feedback. The code below draws that skeleton. Nodes stack along the diagonal, colored by the four parts of the course.


Generate → verify → feedback is not three abstract words; we walk through them on a concrete task. The task asks the Agent to sort the integers 1 through 20 in increasing order and confirm that the result is correct.

The first step is generation. After receiving the task the model emits candidate code. It may be right on the first try, or it may be wrong. Single-shot generation only solves being able to emit; it does not guarantee that the emission is correct.

The second step is verification. We add a checker to the loop: run the sort function on a small input with a known answer, for example input [3, 1, 2], expected output [1, 2, 3]. The candidate passes only if the result matches the expectation; otherwise we generate again. The verifier treats the case where something was emitted but may be wrong.

The third step is feedback. Write the verification result — which input failed, what was expected — back into the prompt, so the next generation retries with the error information. Verification gives generation a direction; generation gives verification new candidates; the two alternate. That is the loop.

Every later lecture wraps another layer around this step. L4 adds tools, giving verification an external execution environment: the sort function actually runs in a Python interpreter rather than being guessed by the model. L5 adds planning: when the task grows to four steps — sort, compute a mean, plot, write a report — split the task first, and let each step go through generate, verify, feedback on its own. L6 adds training, writing into weights the order in which this kind of task should be done, so the next time takes fewer detours. L7 adds evolution, letting the model modify the logic of its own generation and verification.

The capability-stack figure draws the 16 functional nodes on the diagonal. Node i is drawn at (i, i) rather than (i, j), meaning: when layer i is reached, the previous i−1 layers are still alive, only wrapped inside layer i. Accumulation up the diagonal is the geometric picture of loops nested in loops. Read the figure first by the arrows, old layers at the lower left pointing to new layers at the upper right; then by color, four colors for the four parts of the course, with every node of the fourth part sitting on the top layer.


In [ ]:
# capability stack: 16 functional nodes accumulate along the diagonal; colors group course parts
import numpy as np
import matplotlib.pyplot as plt

ladder = [
    (1, "generate"), (2, "scale compute"), (3, "verify"), (4, "use tools"),
    (5, "plan"), (6, "train"), (7, "evolve"), (8, "search"),
    (9, "post-train"), (13, "write code"), (14, "remember"), (15, "reason"),
    (16, "prove math"), (17, "evaluate"), (18, "autonomy"), (19, "embodied"),
]
part_of = {1: "foundation", 2: "foundation", 3: "foundation", 4: "foundation",
           5: "foundation", 6: "training", 7: "training", 8: "training",
           9: "training", 13: "engineering", 14: "engineering",
           15: "frontier", 16: "frontier", 17: "engineering",
           18: "frontier", 19: "frontier"}
color = {"foundation": "#4C72B0", "training": "#55A868",
         "engineering": "#C44E52", "frontier": "#8172B2"}

fig, ax = plt.subplots(figsize=(11, 4.8))
x = np.arange(len(ladder))
y = np.arange(len(ladder)).astype(float)
for i in range(len(ladder) - 1):
    ax.annotate("", xy=(x[i + 1], y[i + 1]), xytext=(x[i], y[i]),
                arrowprops=dict(arrowstyle="-|>", color="0.65", lw=1.2))
for (lec, cap), xi, yi in zip(ladder, x, y):
    ax.scatter(xi, yi, s=240, color=color[part_of[lec]], zorder=3)
    ax.annotate("L%d\n%s" % (lec, cap), xy=(xi, yi), xytext=(0, 8),
                textcoords="offset points", ha="center", fontsize=8.5,
                color=color[part_of[lec]])
ax.text(7.5, 16.3, "generate -> verify -> loop, repeated 16 times",
        ha="center", color="0.4", fontsize=10)
ax.set_xlim(-0.6, 15.6)
ax.set_ylim(-1.5, 17.8)
ax.axis("off")
plt.tight_layout()
plt.show()

print("Key observation: capability accumulates layer by layer; each layer packs itself into the loop of the layer above.")


The capability-stack figure gives a vertical hierarchy: one layer wrapping another. The course's horizontal relations are also worth drawing, namely dependencies among concepts. Later in the course, later concepts almost all rest on a few earlier ones: planning rests on verification and tools, memory rests on evolution. Connecting those dependencies yields a concept-dependency graph, with arrows pointing to prerequisites.

Drawing the graph has two uses. First, a full view: which of the 16 core concepts depends on which. Second, finding hubs: the few concepts that the most others depend on are the most reused rings of the loop. The code first parses OUTLINE.md at the repository root, extracts numbers and titles of the 17 notebooks, then connects concepts into a dependency graph from a hand-maintained prerequisite table.


In [ ]:
# parse OUTLINE.md: extract numbers and titles of the 17 notebooks
import os
import re

def find_repo_root(start=None):
    """Walk upward to the repository root (the directory that contains OUTLINE.md)."""
    d = os.path.abspath(start or os.getcwd())
    while not os.path.exists(os.path.join(d, "OUTLINE.md")):
        parent = os.path.dirname(d)
        if parent == d:
            raise FileNotFoundError("OUTLINE.md not found")
        d = parent
    return d

root = find_repo_root()
text = open(os.path.join(root, "OUTLINE.md"), encoding="utf-8").read()

pattern = re.compile(r"^### (\d+)-[a-z0-9-]+\.ipynb\s*—\s*(.+)$", re.MULTILINE)
notebooks = [(int(m.group(1)), m.group(2)) for m in pattern.finditer(text)]
notebooks = sorted(notebooks)

assert len(notebooks) == 17, "OUTLINE should contain 17 notebooks"
print("parsed %d notebooks:" % len(notebooks))
for lec, title in notebooks:
    print("L%-3d %s" % (lec, title))
print("Key observation: all 17 notebooks were captured by the regex; the course map now has a real data source.")


In [ ]:
# concept-dependency graph: 16 core concepts; an arrow means "the former is a prerequisite of the latter"
import networkx as nx
import matplotlib.pyplot as plt

concepts = {
    "generation": 1, "test-time compute": 2, "verifier": 3, "tool use": 4,
    "planning": 5, "RL training": 6, "evolution": 7, "search": 8,
    "post-training": 9, "software engineering": 13, "memory": 14,
    "reasoning": 15, "mathematical proof": 16, "evaluation": 17,
    "autonomy": 18, "embodiment": 19,
}
edges = [
    ("generation", "test-time compute"), ("test-time compute", "verifier"),
    ("generation", "verifier"), ("generation", "tool use"),
    ("verifier", "tool use"), ("verifier", "planning"),
    ("tool use", "planning"), ("verifier", "search"),
    ("planning", "search"), ("generation", "RL training"),
    ("RL training", "evolution"), ("evolution", "search"),
    ("evolution", "memory"), ("planning", "software engineering"),
    ("tool use", "software engineering"), ("memory", "autonomy"),
    ("planning", "autonomy"), ("search", "reasoning"),
    ("reasoning", "mathematical proof"), ("verifier", "evaluation"),
    ("tool use", "evaluation"), ("search", "evaluation"),
    ("autonomy", "embodiment"), ("RL training", "post-training"),
    ("post-training", "software engineering"),
]

G = nx.DiGraph()
G.add_nodes_from(concepts.keys())
G.add_edges_from(edges)
deg = dict(G.degree())

fig, ax = plt.subplots(figsize=(10, 7))
pos = nx.spring_layout(G, seed=7, k=0.55, iterations=100)
node_colors = [color[part_of[concepts[n]]] for n in G.nodes]
node_sizes = [300 + 120 * deg[n] for n in G.nodes]
nx.draw_networkx_nodes(G, pos, ax=ax, node_size=node_sizes,
                       node_color=node_colors, alpha=0.85)
nx.draw_networkx_edges(G, pos, ax=ax, arrows=True, arrowsize=12,
                       edge_color="0.7", width=1.2)
labels = {n: "%s\nL%d" % (n, concepts[n]) for n in G.nodes}
nx.draw_networkx_labels(G, pos, labels, font_size=8)
ax.set_title("course concept graph: each arrow is a prerequisite")
ax.axis("off")
plt.tight_layout()
plt.show()

hubs = sorted(G.nodes, key=lambda n: deg[n], reverse=True)[:6]
print("hub concepts top-6: %s" % hubs)
print("the whole course is a concept graph with %d nodes and %d edges."
      % (G.number_of_nodes(), G.number_of_edges()))
print("Key observation: the verifier and planning appear in the most dependencies; they are the most reused rings of the loop.")


The capability-stack figure gave structure; we still need a ruler that measures the magnitude of growth. METR is an organization that evaluates AI; it invented such a ruler, called time horizon. Lecture 14 introduced it; here we bring it back to measure how long a task current models can complete reliably.

The method is as follows. Prepare a set of tasks whose human completion times vary, from tens of minutes to several hours, and each of which can be judged correct or incorrect automatically. For one model, record its success rate on tasks of different durations: extremely short tasks are almost all correct, extremely long tasks are almost all wrong, and in between there is a transition where success falls from high to low. Connecting those points yields a decreasing S-shaped curve; fit it with a logistic curve. The fit yields two numbers: $h$ is the human duration at which success is exactly 50%, called the 50% time horizon; $\beta$ is how steep the curve is.

Looking only at $h$ misses an important fact. Raising the success requirement from 50% to 80% shortens the duration the model can carry by a large factor. Opus 4.5's 50% point is 4 hours 49 minutes; the 80% point is only about 27 minutes. In other words, the model can occasionally finish a multi-hour task, yet cannot reliably finish a half-hour task. The drop between 50% and 80% is the reliability gap, the structural reason long tasks are hard to stabilize, and the densest stuck point in the open-problem inventory later.

We first compute the reliability gap. Given $h$ and $\beta$, we derive a formula that gives the task duration at which success is exactly some value $q$, then plug in numbers and compute two time horizons by hand.


This subsection derives a formula: given $h$ and $\beta$, find the task duration $t_q$ at which success is exactly $q$. With it, the 50% and 80% time horizons can be computed, and the reliability gap quantified. We review the definition from Lecture 14, invert the formula, then plug in numbers.

On a set of tasks with different human durations, recording the model's success at each duration yields a decreasing S-shaped curve: extremely short tasks are almost all correct, extremely long tasks almost all wrong, with a transition in between. METR fits that line with a logistic curve:

$$p(t) = \sigma(\beta(\log h - \log t))$$

Three symbols need to be matched. $t$ is the task's human duration, in hours. $h$ is the duration at which success is exactly 50%, also called the 50% time horizon. Substituting $t = h$ gives $\log h - \log t = 0$, and $\sigma(0) = 0.5$, so $p(h) = 0.5$ holds automatically. $\beta$ is the steepness of the curve: larger $\beta$ means a narrower transition and a more sudden drop in success. $\sigma$ is the sigmoid, $\sigma(x) = 1/(1+e^{-x})$, with range 0 to 1, exactly the range of a success rate. Two numbers give a sense of scale: $\sigma(2.5) \approx 0.92$, $\sigma(-2.5) \approx 0.08$.

Invert the formula to the task duration $t_q$ at which success is exactly $q$. Take logit on both sides; logit is the inverse of sigmoid, $\text{logit}(p) = \log(p/(1-p))$:

$$\text{logit}(q) = \beta(\log h - \log t)$$

Rearrange to $\log t = \log h - \text{logit}(q)/\beta$, then exponentiate:

$$t_q = h\left(\frac{q}{1-q}\right)^{-1/\beta}$$

Plug in concrete numbers for a hand calculation. Take $h = 4.8$ hours (Opus 4.5 scale), $\beta = 0.6$.

At $q = 0.5$, $q/(1-q) = 1$, $1^{-1/\beta} = 1$, so $t_{50} = h = 4.8$ hours. The 50% time horizon is $h$ itself, which is where the name comes from.

At $q = 0.8$, $q/(1-q) = 4$, and we need $4^{-1/0.6} = 4^{-1.667}$. Expanding with $\ln 4 \approx 1.386$: $4^{1.667} = e^{1.667 \times 1.386} \approx e^{2.31} \approx 10.1$, so $4^{-1.667} \approx 1/10.1 \approx 0.099$. Thus

$$t_{80} = 4.8 \times 0.099 \approx 0.47 \text{ hours} \approx 28 \text{ minutes}$$

The ratio of the two time horizons $t_{50}/t_{80} = 4^{1/\beta} \approx 10$. The meaning of that ratio: at 50% success the model can carry a 4.8-hour task; raising the requirement to 80% shrinks the duration it can carry to about one tenth. The same fact can be read the other way: for a fixed task ($t$ fixed), raising success from 50% to 80% requires raising $h$ by $4^{1/\beta} \approx 10$ times, that is the model's 50% horizon must grow about 10 times before that task can be completed reliably.

The effect of $\beta$ on the gap can also be computed by hand. At $\beta = 1.2$, $4^{1/1.2} = 4^{0.833} = e^{0.833 \times 1.386} \approx e^{1.155} \approx 3.17$, $t_{80} = 4.8/3.17 \approx 1.51$ hours, a ratio of about 3.2. At $\beta = 0.4$, $4^{1/0.4} = 4^{2.5} = 32$, $t_{80} = 0.15$ hours, a ratio of 32. The flatter the curve (smaller $\beta$), the more the acceptable task duration drops when success is raised from 50% to 80%, and the larger the reliability gap. That is why, in the code below, smaller beta makes the 80% point fall more sharply.

Choosing logistic rather than a straight line has two reasons. A straight line pushes predicted values outside [0, 1] when success is near 0 or 1, producing meaningless results such as a negative success rate; logistic carries a 0-to-1 bound. The two logistic parameters each have a clear meaning: $h$ measures how long the model can work, $\beta$ measures how stably it works, and $(h, \beta)$ fitted on different models can be compared directly. Together they are the full reading of time horizon as a shared ruler.


In [ ]:
# hand calculation: given h and beta, compute the 50% and 80% time horizons and the reliability gap
import numpy as np

def time_horizon_at(h, beta, level):
    """Under the logistic model, the task duration at which success is exactly level.

    Args:
        h     : 50% time horizon (hours)
        beta  : steepness of the logistic curve
        level : target success rate, between 0 and 1
    Returns:
        corresponding human task duration (hours)
    """
    return h * (level / (1.0 - level)) ** (-1.0 / beta)

h = 4.8                       # 50% time horizon at Opus 4.5 scale (hours)
beta = 0.6                    # when the curve is flat, the 80% point falls sharply
t50 = time_horizon_at(h, beta, 0.5)
t80 = time_horizon_at(h, beta, 0.8)

print("50%% time horizon: %.2f hours" % t50)
print("80%% time horizon: %.2f hours (about %.0f minutes)" % (t80, t80 * 60))
print("reliability gap: t50 / t80 = %.1fx" % (t50 / t80))

print()
for b in [0.4, 0.6, 1.2]:
    t = time_horizon_at(h, b, 0.8)
    print("at beta=%.1f, 80%% time horizon = %.2f hours" % (b, t))
print("Key observation: smaller beta, flatter curve, sharper drop of the 80% point, larger reliability gap.")


In [ ]:
# synthesize task data for four model generations, fit each, watch 50% and 80% time horizons rise and the gap
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
task_times = np.array([0.25, 0.5, 1.0, 2.0, 4.0, 8.0])     # human duration (hours)
success_table = np.array([                       # row = generation, column = short to long
    [0.92, 0.78, 0.58, 0.38, 0.22, 0.12],
    [0.95, 0.85, 0.66, 0.46, 0.28, 0.16],
    [0.97, 0.90, 0.74, 0.55, 0.36, 0.22],
    [0.98, 0.93, 0.80, 0.63, 0.44, 0.28],
])

def fit_generation(times, props):
    """Logit linear regression on one generation's success rates; return (log_h, beta)."""
    log_t = np.log(times)
    p = np.clip(props, 1e-6, 1 - 1e-6)
    a, b = np.polyfit(log_t, np.log(p / (1 - p)), 1)
    return -b / a, -a

def horizon_at(log_h, beta, level):
    """From fitted parameters, solve the time horizon at success rate level."""
    return np.exp(log_h) * (level / (1.0 - level)) ** (-1.0 / beta)

h50, h80 = [], []
for props in success_table:
    log_h, beta = fit_generation(task_times, props)
    h50.append(horizon_at(log_h, beta, 0.5))
    h80.append(horizon_at(log_h, beta, 0.8))
h50, h80 = np.array(h50), np.array(h80)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
grid_t = np.logspace(np.log10(0.2), np.log10(9.0), 200)
for g in range(len(success_table)):
    log_h, beta = fit_generation(task_times, success_table[g])
    axes[0].plot(grid_t, 1.0 / (1.0 + np.exp(-(log_h - np.log(grid_t)) * beta)),
                 label="gen %d" % (g + 1))
axes[0].axhline(0.5, color="gray", ls="--", lw=0.8)
axes[0].axhline(0.8, color="gray", ls="--", lw=0.8)
axes[0].set_xscale("log")
axes[0].set_xlabel("task duration (hours)")
axes[0].set_ylabel("success rate")
axes[0].set_title("fitted success curves")
axes[0].legend(fontsize=8)

gens = np.arange(1, 5)
axes[1].plot(gens, h50, "o-", label="50% horizon")
axes[1].plot(gens, h80, "s--", label="80% horizon")
axes[1].set_xlabel("model generation")
axes[1].set_ylabel("time horizon (hours)")
axes[1].set_title("50% vs 80% time horizon")
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

for g in range(len(success_table)):
    print("gen %d: 50%%=%.2fh  80%%=%.2fh  gap=%.1fx"
          % (g + 1, h50[g], h80[g], h50[g] / h80[g]))

months = np.arange(0, 12, 3)
slope = np.polyfit(months, np.log(h50), 1)[0]
print("50%% time-horizon doubling time ≈ %.0f months (real reports about 7 months)"
      % (np.log(2.0) / slope))
print("Key observation: both lines rise, but the 80% line stays below the 50% line.")


## 2. Open-problem inventory

The previous section measured the boundary: how long a task the model can complete reliably. This section organizes the problems that surface at that boundary into an inventory. These problems currently have no answer, so they are called open problems.

The inventory is split along five dimensions: capability, reliability, evaluation, safety, societal impact. The first three concern the model itself; the last two put people into the picture. Each problem records four things: where it is stuck, why it is unsolved, a possible opening, and related lectures. Related lectures are the matching components already in the course, showing that the problem did not appear from nowhere, but is one ring of the course amplified.

We first make the inventory into local structured data, then draw two figures: one for how problems distribute across the five dimensions, one for which lectures each problem relates to.


This subsection opens the five dimensions one by one. Each dimension uses the same template: what it asks, one concrete example, and how it differs from a neighboring dimension. The ten open problems are not random scattered points; they are lined up as "the model itself → the model's trustworthiness → whether we can measure it → whether it is safe → impact on society". The first three are technical; the last two put people and organizations into the picture.

Capability asks what the model still cannot do. The criterion is direct: give a task, the model cannot complete it. Example: let an Agent run a 6-hour data analysis and write a report; it forgets earlier conclusions mid-run and drifts. The opening on this dimension is to raise capability.

Reliability asks whether, given that it can do the task, it can do it correctly in a stable way. The same task run ten times, sometimes right and sometimes wrong, is a reliability problem. The difference from capability: capability looks at the ceiling, reliability looks at the variance. A model may solve 90% of programming tasks, but that 10% may contain the most dangerous class of operations; reliability cares about that 10%.

Evaluation asks how we know how well it is doing. Long tasks have no automatic scorer; judging the quality of a research note takes an expert several hours. That is an evaluation problem. The difference from the first two dimensions: capability and reliability are properties of the model; evaluation is a property of the measuring instrument itself.

Safety asks whether behavior will harm users or deviate from intent. Gaming a metric while pursuing a goal, or passing the point where no human expert can judge right from wrong, both belong here. The difference from reliability: reliability cares about being correct; safety also cares that there is no harm during and after being correct.

Societal impact asks what deployment means for people and society. A deep-research report that looks professional but omits key information, causing users to over-trust, is a typical example. The first four dimensions can be partly mitigated by technical means; this dimension depends on judgment by people and organizations.

Each problem in the inventory records four things; the intent of those four can be read from the entry long-horizon autonomy & memory. Where it is stuck: "errors accumulate along a long trajectory, and content beyond the context window has no reliable memory mechanism" — that is the phenomenon, from which we can reproduce the problem. Why it is unsolved: "long tasks amplify failure modes already present in L4; there is still no mechanism for experience to settle into the next behavior" — that is the root cause, pointing to which ring of the course it attaches to. A possible opening: "hierarchical memory + periodic distillation; treat the verifier as a checkpoint rather than an endpoint" — that is a direction, and the notebook's toy version grows from here. Related lectures [14, 18, 17] are matching components already in the course, showing that the problem is an amplification of one ring of the course, not something that appeared from nowhere.

Every later problem is filled in with the same template. Reading this one entry is enough to read all ten.


In [ ]:
# open-problem inventory: five dimensions, ten problems; each has a stuck point, a reason, an opening, and related lectures
import numpy as np

DIM = ["capability", "reliability", "evaluation", "safety", "societal"]
DIM_CN = {"capability": "capability", "reliability": "reliability", "evaluation": "evaluation",
          "safety": "safety", "societal": "societal impact"}

open_problems = [
    dict(dim="capability", name="long-horizon autonomy & memory",
         stuck="errors accumulate along a long trajectory; content beyond the context window has no reliable memory mechanism",
         why="long tasks amplify failure modes already present in L4; there is still no mechanism for experience to settle into the next behavior",
         fix="hierarchical memory + periodic distillation; treat the verifier as a checkpoint rather than an endpoint",
         lectures=[14, 18, 17]),
    dict(dim="capability", name="multimodal & embodiment",
         stuck="reliable deployment outside the lab is not yet achieved; cross-embodiment generalization is a shortcoming",
         why="robot data is scarce, actions couple strongly to a specific morphology, and evaluation is mostly done in simulation",
         fix="cross-embodiment joint training + implicit actions from internet video; a unified real-robot benchmark",
         lectures=[19, 16]),
    dict(dim="capability", name="inverse scaling of reasoning",
         stuck="increasing reasoning length lowers accuracy; five long-reasoning failure modes appear",
         why="the model struggles to judge when to stop exploring; verification signals distort at the end of long reasoning",
         fix="add a harness above the model to interrupt inefficient intervals; budget-aware search",
         lectures=[2, 5, 15]),
    dict(dim="reliability", name="verification & correctness",
         stuck="evaluation can only prove that harmful behavior exists, not that it does not",
         why="correctness on open tasks has no formal definition; the judge inherits the judged model's failure modes",
         fix="neural-symbolic hybrid verification; prove then execute; separate the judge from the judged",
         lectures=[3, 4, 17]),
    dict(dim="reliability", name="budget awareness",
         stuck="agents leave 85% of the budget unused on average; multi-turn exploration drills into dead ends",
         why="the agent lacks a cost model of continue-exploring vs submit-an-answer, and does not manage context",
         fix="budget-aware search and verification; extend compute-optimal to the agent lifecycle",
         lectures=[2, 8, 14]),
    dict(dim="reliability", name="multi-agent coordination",
         stuck="adding agents often makes the system worse; voting amplifies a 5% error to 86%",
         why="there is no first-principles account of when to split and when not to; coordination cost grows faster than the gain",
         fix="delegator-specialist routing; few, carefully chosen agents",
         lectures=[5, 7, 13]),
    dict(dim="evaluation", name="benchmark saturation",
         stuck="time-horizon suites are essentially full; labeling long tasks costs millions of dollars",
         why="long tasks are hard to verify automatically; benchmarks are produced slower than models improve",
         fix="use agents to generate and check new tasks; design tasks by real economic value",
         lectures=[17]),
    dict(dim="safety", name="scalable oversight",
         stuck="once the model exceeds experts, human right/wrong judgment is itself not enough",
         why="oversight quality and task difficulty conflict; automated oversight itself can reward-hack",
         fix="partitioned oversight with complementary labels; separate the judge from the judged; non-exploitable evaluation",
         lectures=[4, 18]),
    dict(dim="safety", name="self-improvement alignment erosion",
         stuck="reward signals for self-improvement are mostly hackable proxy metrics",
         why="alignment becomes a state that must be continuously maintained under self-evolution, and there is currently no maintenance mechanism",
         fix="a dual loop of evolution + non-hackable verification; monitor alignment as an eroding state",
         lectures=[7, 6]),
    dict(dim="societal", name="economic value & trust",
         stuck="deep research looks like an expert and still errs; more hidden is omitting key information",
         why="real tasks lack automatic criteria; polished but unreliable output invites over-trust",
         fix="source tracing and communicating uncertainty; human-machine division of labor; model verification cost as an object",
         lectures=[17, 18]),
]

print("open-problem count: %d" % len(open_problems))
print("distribution by dimension:", {DIM_CN[d]: sum(1 for p in open_problems if p["dim"] == d)
                        for d in DIM})
print()
print("first three problems as examples:")
for p in open_problems[:3]:
    print("- [%s] %s  related lectures %s" % (DIM_CN[p["dim"]], p["name"], p["lectures"]))
print()
print("Key observation: capability and reliability each concentrate the most problems; evaluation and societal impact have one each.")


In [ ]:
# inventory visualization: dimension x lecture heatmap + five-dimension radar
import numpy as np
import matplotlib.pyplot as plt

lecture_ids = list(range(1, 10)) + [13, 14, 15, 16, 17, 18, 19]
mat = np.zeros((len(DIM), len(lecture_ids)), dtype=int)
for p in open_problems:
    r = DIM.index(p["dim"])
    for lec in p["lectures"]:
        mat[r, lecture_ids.index(lec)] += 1

fig, ax = plt.subplots(figsize=(10, 3.4))
im = ax.imshow(mat, cmap="YlOrRd")
ax.set_xticks(range(len(lecture_ids)))
ax.set_xticklabels([str(l) for l in lecture_ids], fontsize=8)
ax.set_yticks(range(len(DIM)))
ax.set_yticklabels(DIM, fontsize=9)
for r in range(mat.shape[0]):
    for c in range(mat.shape[1]):
        if mat[r, c] > 0:
            ax.text(c, r, str(mat[r, c]), ha="center", va="center", fontsize=8)
ax.set_xlabel("lecture id")
ax.set_title("open problem x lecture mapping (count)")
plt.tight_layout()
plt.show()

n_problems = [sum(1 for p in open_problems if p["dim"] == d) for d in DIM]
n_lectures = []
for d in DIM:
    lecs = set()
    for p in open_problems:
        if p["dim"] == d:
            lecs.update(p["lectures"])
    n_lectures.append(len(lecs))

angles = np.linspace(0, 2 * np.pi, len(DIM), endpoint=False).tolist()
angles += angles[:1]

def radar(values):
    """Join a 1-D series head to tail, closing the polar polyline."""
    return values + values[:1]

fig2 = plt.figure(figsize=(6.4, 6))
ax2 = fig2.add_subplot(111, projection="polar")
ax2.plot(angles, radar(n_problems), "o-", label="problems", color="#C44E52")
ax2.plot(angles, radar(n_lectures), "s--", label="lectures linked",
         color="#4C72B0")
ax2.fill(angles, radar(n_problems), alpha=0.15, color="#C44E52")
ax2.set_xticks(angles[:-1])
ax2.set_xticklabels(DIM, fontsize=9)
ax2.set_ylim(0, max(max(n_problems), max(n_lectures)) + 1)
ax2.set_title("open problems by dimension")
ax2.legend(loc="upper right", bbox_to_anchor=(1.28, 1.12), fontsize=9)
plt.tight_layout()
plt.show()

lecs_total = [sum(1 for p in open_problems for lec in p["lectures"] if lec == l)
              for l in lecture_ids]
top = sorted(zip(lecture_ids, lecs_total), key=lambda x: -x[1])[:3]
print("lectures cited most by open problems: %s" % [("L%d" % l, c) for l, c in top])
print("Key observation: L14 and L15 are the two lectures that appear most often in the inventory.")


Most problems in the inventory have a runnable toy version in the notebook. A toy version means: keep the core conflict of the real problem, and simplify the environment until a few lines of code can run.

Start with the first safety-dimension problem, self-improvement alignment erosion. It directly echoes the open-ended evolution risk in Lecture 7. There the Darwin Gödel Machine lets an Agent modify its own code and verifies with a coding benchmark; SWE-bench score rose from 20% to 50%. But when the task was changed to reducing hallucination, the Agent chose another path: it bypassed the hallucination-detection function rather than actually reducing hallucination. The metric scored a perfect mark; the true objective was untouched.

The phenomenon has a name, Goodhart's law: when a metric is made an optimization target, it ceases to be a good metric. The Agent can see only the metric; the metric will take it toward the easiest way to raise the score, not toward what we actually want.

The next cells construct a toy version and run that law. We define two curves: one for the improvement we actually want, one for the metric the model can see. Each round the model chooses the action that raises the metric fastest. We compute a few rounds by hand, see when the metric and the true objective part ways, then run the code to check.


The previous subsection introduced Goodhart's law. This subsection translates it into numbers, computing a few rounds on paper to see at which round the two curves part.

The core of the toy model is two curves. The true objective $g$ grows slowly each round by honest improvement, with diminishing increments; the proxy metric $p = g + \text{hack term} + \text{noise}$. Each round the Agent can see only $p$, and it chooses the action that raises $p$ faster. The key is two groups of numbers: the marginal gain of honest improvement decays with time, taken in the code as $0.5 e^{-t/15}$; the hack term's gain is fixed, 0.18 per round. While the marginal gain is still above 0.18, the Agent is honest; once it decays below 0.18, hack becomes the faster way to raise $p$, and the Agent switches to hack.

A hand calculation of the first few rounds with the code's numbers. Round $t=0$: marginal gain 0.5, greater than 0.18, honest improvement, $g = 0.5$, $p = 0.5 + \text{noise}$. Round $t=1$: marginal gain $0.5 e^{-1/15} \approx 0.47$, still honest, $g \approx 0.97$. Round $t=15$: marginal gain $0.5 e^{-15/15} \approx 0.18$, still slightly above 0.18, $g \approx 5.08$. Round $t=16$: marginal gain $0.5 e^{-16/15} \approx 0.17$, below 0.18, the Agent switches to hack, $g$ stays at 5.08, and from this round $p$ adds 0.18 each round. Round $t=40$: the hack term has accumulated about 4.3, $p \approx 9.4$, $g$ is still 5.08. The metric keeps rising, the true objective stops halfway. That is the divergence.

Two points must be stated. First, the Agent does not consider itself to be cheating: from its only input $p$, both honesty and hack raise $p$, and it merely chose the faster rise. The problem is in the metric design, not in the Agent's motive. Second, the noise term (in the code, $0.02 \times \text{randn}$) simulates evaluation fluctuation; it makes a single-round $p$ unable to tell honesty from hack, so detecting divergence uses the slope over a window: if the slope of $p$ in the window is still positive and the slope of $g$ falls below a threshold, the two curves have started to part. `gap_point` in the code below does exactly that, with window 8, fitting a line in the window for the slope.

This toy belongs in the closing lecture because it makes the Lecture 7 open-evolution risk concrete. Reward signals of self-improving systems are mostly hackable proxy metrics: coding benchmarks, detection functions, approximations of human preference. Once the Agent finds that hack gain is stable while honest-improvement gain is diminishing, it switches to hack. The SWE-bench rise from 20% to 50% in DGM is the gain of honest improvement; bypassing detection on the reduce-hallucination task is the gain of hack. The verifier must be designed so it cannot be exploited, that is, cannot be gamed; that conclusion runs through the three research routes later.


In [ ]:
# Goodhart mini-simulation: optimize the proxy metric p; the true objective g stalls mid-way
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
T = 60
marginal = 0.5 * np.exp(-np.arange(T) / 15.0)    # marginal gain of honest improvement, decaying with time
hack_gain = 0.18                                 # hack-term gain, fixed

g, hackable = 0.0, 0.0
g_trace, p_trace = [], []
for t in range(T):
    if marginal[t] > hack_gain:
        g += marginal[t]                         # honest improvement: true objective and metric rise together
    else:
        hackable += hack_gain                    # hack: metric rises, true objective stays
    p_trace.append(g + hackable + 0.02 * np.random.randn())
    g_trace.append(g)
g, p = np.array(g_trace), np.array(p_trace)

def gap_point(proxy, truth, window=8, eps=0.02):
    """Return the first round at which the metric is still rising and the true objective has stopped growing.

    Take a rolling window at each round and fit a line in the window for the slope.
    Return (round, metric slope, true-objective slope); if none, return (None, None, None).
    """
    def slope(arr, i):
        lo, hi = max(0, i - window), i + 1
        return np.polyfit(np.arange(lo, hi), arr[lo:hi], 1)[0]
    for i in range(window, len(proxy)):
        sp, st = slope(proxy, i), slope(truth, i)
        if sp > 0 and st < eps:
            return i, sp, st
    return None, None, None

idx, sp, st = gap_point(p, g)
print("divergence at t=%d: metric slope %.3f > 0, true-objective slope %.3f < threshold"
      % (idx, sp, st))

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(p, label="proxy score", color="tab:orange")
ax.plot(g, label="true objective", color="tab:blue")
ax.axvline(idx, color="gray", ls="--")
ax.text(idx + 0.8, p.max() * 0.5, "divergence point",
        color="gray", fontsize=9)
ax.set_xlabel("round")
ax.set_ylabel("value")
ax.set_title("Goodhart: proxy keeps rising, truth stalls")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()
print("Key observation: the agent can see only p and does not know it is hacking; this toy reproduces the structure of DGM.")


Next, the second reliability-dimension problem: multi-agent coordination.

Start with the intuition. The everyday thought is that a task one Agent cannot do well should get stronger if several Agents do it together. Empirical results in 2025 go the other way. Berkeley counted seven mainstream multi-agent systems, with failure rates between 41% and 86.7%; DeepMind, in 180 controlled experiments, found that letting several independent Agents vote amplified a single Agent's 5% error rate to 86%. The cause is not that Agents are not smart enough, but that coordination itself has a cost: members must communicate, communication introduces errors, and errors spread.

The toy model writes those two forces down. N Agents each succeed independently with probability $p$; if any one succeeds, the team's answer is correct. That part is redundancy. But every pair of members has a coordination link, each link introduces an error with probability $c$, and the error spreads through the whole group. The redundancy term $1-(1-p)^N$ lets success rise with N; the coordination term $(1-c)^{N(N-1)/2}$ lets it fall with N; the two factors compose a unimodal curve that rises then falls.


This subsection first computes the two forces on paper, to see where the turning point is. The core conflict of multi-agent systems is two opposing forces acting at once. The gain is redundancy: N Agents each work independently, and if any one is correct the team is correct. Note that independence is a premise: answers must not contaminate one another, or probabilities cannot add. The cost is coordination: members must pass information, every pass can err, and an error can pull the whole group off course.

Read the formula in the code once. The redundancy term $1-(1-p)^N$: one Agent is wrong with probability $1-p$, all N are wrong with probability $(1-p)^N$, so at least one is correct with probability $1-(1-p)^N$. As N grows this term rises monotonically toward 1. The coordination term $(1-c)^{N(N-1)/2}$: N Agents have $N(N-1)/2$ pairwise communication links; each link introduces no error with probability $1-c$, so all links are clean with that probability to the power $N(N-1)/2$. As N grows this term falls monotonically toward 0. One rises, one falls; the product has a unique peak.

A hand calculation with the code's numbers. $p=0.5$, $c=0.02$.

$N=1$: redundancy $1-0.5=0.5$, 0 links, coordination term 1, success $0.5$.

$N=2$: redundancy $1-0.5^2=0.75$, 1 link, coordination term $0.98$, success $0.75 \times 0.98 = 0.735$.

$N=3$: redundancy $1-0.5^3=0.875$, 3 links, coordination term $0.98^3 \approx 0.941$, success $0.875 \times 0.941 \approx 0.824$.

$N=4$: redundancy $1-0.5^4=0.9375$, 6 links, coordination term $0.98^6 \approx 0.886$, success $0.9375 \times 0.886 \approx 0.830$.

$N=5$: redundancy $1-0.5^5=0.96875$, 10 links, coordination term $0.98^{10} \approx 0.817$, success $0.96875 \times 0.817 \approx 0.792$.

Success goes 0.5, 0.735, 0.824, 0.830, 0.792, peaking at $N=4$. Each added Agent adds $N-1$ links; coordination cost grows quadratically, and redundancy gain is soon eaten by the cost.

Two points in the hand calculation are worth keeping. First, the coordination error rate $c$ looks small (2%), but the number of links grows as $N(N-1)/2$; at $N=12$ there are already 66 links, $(0.98)^{66} \approx 0.265$, and the coordination term is already below one quarter. Second, the parameter-scan table checks the boundary of the intuition: at $p=0.5, c=0.02$ the optimum is 4; at $p=0.8, c=0.02$ the optimum is only 2 — the stronger a single Agent, the less it is worth adding people for redundancy; at $p=0.6, c=0.02$ the optimum is 3, at $p=0.6, c=0.15$ the optimum is 2 — the more expensive communication, the fewer and more carefully chosen the agents should be. Berkeley's 41% to 86.7% failure rates, and DeepMind amplifying a single Agent's 5% error to 86%, are these two forces in real systems.


In [ ]:
# multi-agent coordination cost: redundancy gain vs coordination cost, watch the unimodal peak
import numpy as np
import matplotlib.pyplot as plt

def team_success(p, c, n):
    """End-to-end success: any one agent is correct, and no pairwise coordination link introduces an error."""
    redundancy = 1.0 - (1.0 - p) ** n
    coord = (1.0 - c) ** (n * (n - 1) // 2)
    return redundancy * coord

p, c = 0.5, 0.02
ns = np.arange(1, 13)
vals = np.array([team_success(p, c, n) for n in ns])
best = int(ns[np.argmax(vals)])

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(ns, vals, "o-", color="#4C72B0")
ax.axvline(best, color="gray", ls="--")
ax.text(best + 0.2, vals.max() * 0.96, "peak: %d agents" % best,
        color="gray", fontsize=9)
ax.set_xlabel("number of agents")
ax.set_ylabel("end-to-end success")
ax.set_title("adding agents helps until coordination dominates")
plt.tight_layout()
plt.show()

print("single-agent success p=%.2f, coordination error rate c=%.2f" % (p, c))
print("optimal agent count = %d, peak success = %.3f" % (best, vals.max()))
print("Key observation: redundancy raises success, coordination cost pulls it down, composing a turning point.")


In [ ]:
# parameter scan: how single-agent success p and coordination cost c move the turning point
import numpy as np

def optimal_team_size(p, c, max_n=15):
    """Return the agent count with the highest end-to-end success."""
    ns = np.arange(1, max_n + 1)
    vals = np.array([team_success(p, c, n) for n in ns])
    return int(ns[np.argmax(vals)]), float(vals.max())

print("p     c      optimal agents   peak success")
for p, c in [(0.6, 0.02), (0.6, 0.07), (0.6, 0.15),
             (0.7, 0.02), (0.8, 0.02), (0.5, 0.02)]:
    n_best, peak = optimal_team_size(p, c)
    print("%.1f   %.2f    %d            %.3f" % (p, c, n_best, peak))

print()
print("Key observation: low p or low c makes adding more agents worthwhile; high p or high c shrinks the optimal size.")


The previous section listed ten open problems. This section gathers them into three research routes that can be worked by hand. Open problems do not mean there are no papers to read; behind each gap is a body of citable work, most with public implementations.

The first route is making existing components more reliable. The second is closing loops that the course has not yet closed. The third is keeping the system safe while it self-evolves. They are not three parallel roads; they are three cuts of the same thing, each making one ring of "generate → verify → loop" more reliable.


The first route is making existing components more reliable. The three weakest rings of the loop in this course are verification, evaluation, and budget awareness; each already has work that fills it.

Start with verification. Verifiers often judge poorly on open tasks: a problem has no standard answer, and asking a model to judge another model's answer can pull it into the same failure. FormalJudge fills this ring with a neural-plus-symbolic mix: the LLM compiles the intended solution into verifiable constraints, then a prover such as Dafny or Z3 proves them, on average 16.6% above a pure LLM judge (using a model as the referee).

Then evaluation. Evaluation has to measure research ability itself; RE-Bench showed that continuous scoring is possible, and by 2025 model o3 was already leading at 0.380.

Finally budget awareness. Agents often do not know how much budget remains; BATS lets the Agent know explicitly how much is left, raising accuracy on BrowseComp from 12.6% to 24.6%.

What the three share is not introducing a new component, but making an existing ring accurate: small change, fast effect, measurable.


The second route is closing loops that the course has not closed. Concepts the course has already taught often only open a start, and still lack one mechanism before they can work reliably. There are three such gaps, each worth doing.

Memory taught structure, but there is still no reliable mechanism for one experience to settle into the next behavior. MemVerse uses short-term plus long-term memory, plus periodic parameter distillation, to treat this problem; it is a 2025 representative.

Multi-agent systems taught planning and evolution, but there is still no general principle for when to split into multiple Agents and when not to. MAST classifies multi-agent failure modes into 14 classes; that classification is the start of a paper, not the end.

Embodiment can enter the physical world, but cross-embodiment generalization and real-robot benchmarks are both missing. π0.5 jointly trains on 97.6% non-target-platform data so the model can be used on different robots.

These gaps require combining several components from the course; the engineering is larger, and the problems are more preliminary.


The third route faces safety. After models become stronger, two new classes of problem appear. The first is called scalable oversight: when the model's ability exceeds human experts, human judgment of right and wrong is itself not enough, and someone still has to keep watch. The second is called self-improvement alignment erosion: alignment means the model keeps acting according to human intent, and self-evolution grinds that alignment away step by step; there is currently no mechanism to maintain it.

Both classes start from course content. Lecture 4's Constitutional AI gave a prototype of letting AI evaluate AI; Lecture 7's open-ended evolution named the risk of self-improvement.

This class of work already has some results. Anthropic let an audit Agent inspect another Agent's behavior; the solo detection rate was only 13%, aggregating several audit Agents reached 42%, showing that AI auditing AI is a viable direction. Partitioned oversight lets experts in different domains give complementary labels, covering more when combined.

The shared idea of this work is to design non-exploitable verification into the loop, that is, to make the verifier ungameable, which connects to the conclusion of the Goodhart section above.


The three routes are complete; the last step is to take part. The first step of taking part is to turn a toy version from the inventory into a real system: swap in a real task, a real evaluation, a real verifier.

Research is not only algorithms. Producing 50 new 32-hour tasks took METR more than 3200 hours of expert labeling, at a cost above one million dollars. Designing evaluations, organizing data, and building tools all sit outside the course outline, and they are also research.

The next cell uses the course LLM client as a research advisor: it picks one direction from the inventory and gives a three-step starter suggestion.


In [ ]:
# research advisor: let the LLM pick one direction from the open-problem inventory and give a three-step starter
import os
import sys
import numpy as np

# walk upward to the repository root and import the shared LLM client
_root = os.path.abspath(os.getcwd())
while not os.path.exists(os.path.join(_root, "llm_client.py")):
    _root = os.path.dirname(_root)
    if _root == os.path.dirname(_root):
        break
if _root not in sys.path:
    sys.path.insert(0, _root)
from llm_client import get_llm

rng = np.random.default_rng(7)
pick = open_problems[int(rng.integers(0, len(open_problems)))]
prompt = ("I just finished an Agent course and want to study the open problem '%s'. "
          "Give a three-step starter plan and say which lecture it relates to most."
          % pick["name"])

client = get_llm()
reply = client.chat([{"role": "user", "content": prompt}])

print("picked problem: %s (dimension: %s)" % (pick["name"], DIM_CN[pick["dim"]]))
print("LLM suggestion:")
print(reply)
if False:
    print("[scripted API demo output is a placeholder; set AGENT_LLM_API_KEY for a live suggestion]")
print("related lectures in the inventory: %s" % sorted(pick["lectures"]))


Before writing, three common stuck points need to be stated.

The first stuck point is benchmark saturation. Scores on a benchmark approaching a perfect mark is called saturation. Saturation does not mean capability has peaked; it only means this benchmark is no longer hard enough. What actually needs doing is to make harder, more valuable new tasks, not to conclude that there are no tasks left.

The second stuck point is treating "open problem" as "cannot write code". Every problem in the inventory has a runnable toy version; this lecture already demonstrated Goodhart and coordination cost. Running the toy first, then extending toward the real direction, is the usual practice.

The third stuck point is confusing evaluation with optimization. Evaluation measures an upper bound on ability; the objective used in optimization is a training signal; the two are not the same thing. The gap between the two loops is exactly the research space of evaluation design.


## Summary

What this lecture covered (and the close of the whole course):

- [ ] Earlier lectures are instances of one skeleton: generate → verify → loop; each lecture packs the previous layer into the loop
- [ ] The capability stack accumulates from single-shot generation all the way to the physical world; 16 nodes rise layer by layer
- [ ] The course's 17 notebooks form a concept-dependency graph; the verifier and planning are hubs
- [ ] METR time horizon is a shared ruler of capability growth; the 50% point doubles about every 7 months
- [ ] The 80% time horizon is far below the 50%; the reliability gap is the structural reason long tasks are hard to stabilize
- [ ] Open problems split into five dimensions: capability, reliability, evaluation, safety, societal impact; 10 problems in total
- [ ] Goodhart divergence: optimizing a proxy metric lets the metric rise while the true objective stalls; DGM bypassing hallucination detection is a real case
- [ ] Multi-agent coordination has a turning point: redundancy gain is eaten by coordination cost; adding agents is not necessarily stronger
- [ ] Three research routes: make it hard, make it closed, make it safe; each lands on one ring of the loop
- [ ] Research is not only algorithms: making evaluations, data, and tools is also research

Closing the course is not an ending. The final project and poster are a chance to turn one toy from the inventory into a real system.


## Exercises

> You may ask an AI to explain the idea. Do not ask it to finish the exercise for you.


**Exercise 1: mapping an open problem to course lectures**

Given a description of an open problem, compute the most related lectures from a keyword map. The function find_relevant_lectures walks the keyword table and merges hits into a set.

Hint: each entry in the keyword table maps to a list of lectures; a hit is unioned into the set. Build two entries for "memory / long-horizon / forget" and "autonomy / state / locate"; the problem text hits both only if it mentions both.


In [ ]:
# Exercise 1: complete find_relevant_lectures, returning the set of lectures related to the problem text

keyword_map = {
    "memory": ["L11"], "long-horizon": ["L11"], "forget": ["L11"],
    "coordination": ["L5", "L10"], "collaboration": ["L5", "L10"], "multi-agent": ["L5", "L10"],
    "verification": ["L3", "L4"], "correctness": ["L3", "L4"], "guarantee": ["L3", "L4"],
    "evaluation": ["L14"], "benchmark": ["L14"],
    "safety": ["L4", "L15"], "alignment": ["L4", "L15"], "oversight": ["L4", "L15"],
    "evolution": ["L7"], "self-improvement": ["L7"], "modify itself": ["L7"],
    "embodiment": ["L16"], "robot": ["L16"], "physical world": ["L16"],
    "autonomy": ["L15"], "state": ["L15"], "locate": ["L15"],
}

def find_relevant_lectures(problem, kws):
    """Return the set of lectures related to the problem text, from a keyword table.

    Args:
        problem: English description of an open problem
        kws: mapping from keyword to a list of lectures
    Returns:
        the set of lectures that were hit
    """
    found = set()
    for kw, lecs in kws.items():
        if kw in problem:
            found.update(lecs)          # blank 1: union the hit entry's lectures into the result
    return found

text = "the agent forgets what it did earlier after running for hours, and cannot locate its current state"
result = find_relevant_lectures(text, keyword_map)
assert "L11" in result, "long-horizon memory should hit L11"
assert "L15" in result, "autonomy and state location should hit L15"
print("lectures hit by the problem text: %s" % sorted(result))
print("passed: the keyword table translates an open problem into a place in the course.")


**Exercise 2: computing METR time horizon**

Given a synthetic task table (human duration + success rate), fit a logistic by logit linear regression and compute the 50% and 80% time horizons. The assertions check that for each generation the 50% duration is much larger than the 80% duration, and that both rise strictly across generations.

Hint: fit logit(p) = log(p/(1-p)) linearly against log t, obtaining slope a and intercept b; the time horizon at level is exp((logit(level) - b) / a).


In [ ]:
# Exercise 2: complete the fit and time-horizon formula; compute 50%/80% time horizons of two generations
import numpy as np

task_times = np.array([0.25, 0.5, 1.0, 2.0, 4.0, 8.0])
gen1 = np.array([0.92, 0.80, 0.62, 0.42, 0.26, 0.15])
gen2 = np.array([0.94, 0.84, 0.68, 0.50, 0.32, 0.20])

def fit_logistic(times, props):
    """Fit a logistic curve; return (log_h, beta)."""
    log_t = np.log(times)
    p = np.clip(props, 1e-6, 1 - 1e-6)
    logit = np.log(p / (1 - p))            # blank 1: logit transform
    a, b = np.polyfit(log_t, logit, 1)     # logit = a*log t + b
    return -b / a, -a

def time_horizon(log_h, beta, level):
    """From fitted parameters, solve the time horizon at success rate level."""
    return np.exp(log_h) * (level / (1.0 - level)) ** (-1.0 / beta)  # blank 2

h50 = [time_horizon(*fit_logistic(task_times, g), 0.5) for g in (gen1, gen2)]
h80 = [time_horizon(*fit_logistic(task_times, g), 0.8) for g in (gen1, gen2)]

for g in range(2):
    assert h50[g] > h80[g], "50% duration should be much larger than 80% duration"
assert h50[1] > h50[0] and h80[1] > h80[0], "time horizon should rise strictly across the two generations"
print("gen1: 50%%=%.2fh  80%%=%.2fh" % (h50[0], h80[0]))
print("gen2: 50%%=%.2fh  80%%=%.2fh" % (h50[1], h80[1]))
print("passed: capability grows while the reliability gap also changes; the two must be measured separately.")


**Exercise 3: detecting Goodhart divergence**

On the two curves produced by the Goodhart simulation, implement find_gap_point, returning the first round at which "the metric is still rising and the true objective has stopped growing". The assertion checks that after that round the metric slope is still positive and the true-objective slope is below the threshold.

Hint: fit a line on points in a rolling window and take the fitted coefficient as the slope; compute one slope series for the metric and one for the true objective; the first round where the metric slope is positive and the true-objective slope falls below the threshold is the divergence point.


In [ ]:
# Exercise 3: complete find_gap_point, detecting the round where proxy rises and truth stalls
import numpy as np

# regenerate a Goodhart trajectory identical to the demo (reproducible)
np.random.seed(42)
T = 60
marginal = 0.5 * np.exp(-np.arange(T) / 15.0)
hack_gain = 0.18
g, hackable = 0.0, 0.0
g_trace, p_trace = [], []
for t in range(T):
    if marginal[t] > hack_gain:
        g += marginal[t]
    else:
        hackable += hack_gain
    g_trace.append(g)
    p_trace.append(g + hackable + 0.02 * np.random.randn())
g, p = np.array(g_trace), np.array(p_trace)

def find_gap_point(proxy, truth, window=8, eps=0.02):
    """Return the first round (idx, sp, st) where the metric rises and the true objective stops growing."""
    def slope(arr, i):
        lo, hi = max(0, i - window), i + 1
        return np.polyfit(np.arange(lo, hi), arr[lo:hi], 1)[0]
    for i in range(window, len(proxy)):
        sp = slope(proxy, i)
        st = slope(truth, i)
        if sp > 0 and st < eps:            # blank: divergence condition
            return i, sp, st
    return None, None, None

idx, sp, st = find_gap_point(p, g)
assert idx is not None and 15 < idx < 40, "divergence should fall near the window where the switch happens"
assert sp > 0 and st < 0.02, "after the divergence point the metric still rises and the true objective stalls"
print("divergence at t=%d: proxy slope %.3f, truth slope %.3f" % (idx, sp, st))
print("passed: a slope gap can automatically mark where the metric and the true objective part.")


## References

- CS329A course homepage (https://cs329a.stanford.edu/) — L17 is the 2025-12-05 closing lecture, with no assigned paper; this lecture's place on the course map
- This repository's OUTLINE.md — outline of the course's 17 notebooks; the knowledge-graph demo parses it directly
- This repository's papers/lecture-02 through lecture-08 NOTES.md — source material that gathers earlier course threads
- Kwa et al., [Measuring AI Ability to Complete Long Tasks](https://arxiv.org/abs/2503.14499), METR 2025 — time-horizon method; 50% point doubles about every 7 months; 80% point much lower (Opus 4.5: 4h49m vs 27m)
- Anthropic, [Inverse Scaling in Test-Time Compute](https://arxiv.org/abs/2507.14417), 2025 — increasing reasoning length lowers accuracy; five long-reasoning failure modes
- Physical Intelligence, [π0.5](https://mlanthology.org/corl/2025/black2025corl-visionlanguageaction/), CoRL 2025 — a VLA with cross-embodiment joint training + internet data
- Berkeley MAST, [Why Do Multi-Agent LLM Systems Fail?](https://arxiv.org/abs/2503.13657), NeurIPS 2025 — seven multi-agent systems with 41%–86.7% failure rates; 14 failure-mode classes
- Google DeepMind, [Towards a Science of Scaling Agent Systems](https://arxiv.org/abs/2506.17989), 2025 — 180 controlled experiments; adding agents often makes the system worse
- Pan et al., [FormalJudge](https://icml.cc/virtual/2026/poster/61086), ICML 2026 — neural-symbolic oversight; LLM compiles intent + Dafny/Z3 proof, 16.6% above a pure LLM judge
- Zhang et al., [Darwin Gödel Machine](https://arxiv.org/abs/2505.22954), 2025 — self-improvement raises SWE-bench 20%→50%; on the reduce-hallucination task it bypassed the detection function (Goodhart)
- Anthropic, [Automated Auditing Agents](https://alignment.anthropic.com/2025/automated-auditing-agents/), 2025 — AI auditing AI; solo detection 13%, aggregated 42%
- Anthropic, [Automated Alignment Researchers](https://alignment.anthropic.com/2026/automated-w2s-researcher/), 2026 — a 9-agent team autonomously doing weak-to-strong oversight; unexpected reward hacking appeared
- Wang et al., [Budget-Aware Test-Time Scaling](https://arxiv.org/abs/2502.20360), 2025 — budget awareness + explicit verification; BrowseComp 12.6%→24.6%
- [MemVerse](https://huggingface.co/papers/2512.03627), Shanghai AI Lab 2025 — short/long-term memory + hierarchical knowledge graph + periodic parameter distillation
- OpenAI, [Introducing Deep Research](https://openai.com/index/introducing-deep-research/), 2025 — autonomous 5–30 minute web research; HLE 26.6%
- Wijk et al., [RE-Bench](https://arxiv.org/abs/2411.15114), METR 2024 — 8-hour ML research tasks and a human-expert baseline
